# 1. Ingestion of Crop Data

## Thoughts

- Think the crop data may only be avilable as csv downloads.
- Might think to automate download, but in the first instance, just download and process.

## Data considerations

Crop yield data is available for several crops, over many years, by each region.
Paired data is for weather at a regional level.

## Region Labels

### Whole List

|ID|Crop Regions|
|--|------------|
|1|United Kingdom|
|2|England|
|3|North East|
|4|North West and Merseyside|
|5|Yorkshire & The Humber|
|6|East Midlands|
|7|West Midlands|
|8|Eastern|
|9|South East and London|
|10|South West|
|11|Wales|
|12|Scotland|
|13|Northern Ireland|

#### Countries

|ID|Crop Regions|
|--|------------|
|1|United Kingdom|
|2|England|
|11|Wales|
|12|Scotland|
|13|Northern Ireland|

#### Regions

|ID|Crop Regions|
|--|------------|
|3|North East|
|4|North West and Merseyside|
|5|Yorkshire & The Humber|
|6|East Midlands|
|7|West Midlands|
|8|Eastern|
|9|South East and London|
|10|South West|

# 1. Plan

- Download sample data.
- Ingestion into dataframe.

## Attempt direct ingestion

Try following url:

<https://assets.publishing.service.gov.uk/media/68e778c1e5f463a62cb98588/UK-cops-webseries-251009a.ods>

### Test kernel

In [ ]:
print("hello")

### Get data from url

Get just the data as a respone object.

In [ ]:
# N.B. Need to ensure odfpy is installed in the environment. This allows the ODS binary to be parsed.

import requests
import io
import pandas as pd

url = "https://assets.publishing.service.gov.uk/media/68e778c1e5f463a62cb98588/UK-cops-webseries-251009a.ods"
response = requests.get(url)
response.raise_for_status() # This gives the REST request response.

Option to use the pandas.read_excel function.

In [ ]:
# show sheet names
# https://pandas.pydata.org/docs/dev/reference/api/pandas.ExcelFile.sheet_names.html
crops_file = pd.ExcelFile(io.BytesIO(response.content))
sheets = crops_file.sheet_names

print(sheets)

In [ ]:
# https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html
df_crops = pd.read_excel(io.BytesIO(response.content), engine="odf", header=None)

print(f'Dataframe head data is:\n{df_crops.head()}')

### Read into Dataframe

Get data from specific worksheet with the data

In [ ]:
df_wheat = pd.read_excel(
    io.BytesIO(response.content),
    sheet_name="Regional_wheat",
    engine="odf", header=None
)

print(df_wheat.head())

Get just the rows that show the Yield Data

In [ ]:
df_wheat_yields = df_wheat.loc[17:30]
print(df_wheat_yields.head())

In [ ]:
df_wheat_yields.columns = df_wheat_yields.iloc[0] 	# specifies row 0, first row, as being the column names.
df_yields = df_wheat_yields[1:]						# resests the dataframe data as starting from the second row, index 1 onwards ':'.
df_yields = df_yields.reset_index(drop=True)		# index needs to be reset as the columns row is now the header.

In [ ]:
print(df_yields)

### Remove columns

Drop incomplete data from 2025, and last two columns showing summary statistics

In [ ]:
# data frame info
df_yields.info(verbose=True)
# print(df_yields)

In [ ]:
df_yields.columns

#### Trying to remove columns that are not relevant.

- 2025 is not a complete year.
  - Remove this but note future implementation may want to dynamically identify the current year and remove that only. e.g. 2026 will want 2025 data.
- Neitter '% change 2025/2024' nor '2025 confidence interval' are relevant for the analysis.
  - **N.B.** justify why.

In [ ]:
columns_to_drop = [2025, '% change 2025/2024', '2025 confidence interval']
df_yields.drop(columns_to_drop, axis=1)
df_yields.columns